In [12]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml
import sys
sys.path.append('..')

In [13]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_mensajeria = config['MENSAJERIA_OLTP']
    config_etl = config['ETL_PROCESS']

url_mensajeria = (f"{config_mensajeria['drivername']}://{config_mensajeria['user']}:{config_mensajeria['password']}@{config_mensajeria['host']}:"
          f"{config_mensajeria['port']}/{config_mensajeria['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")

mensajeria = create_engine(url_mensajeria)
etl_conn = create_engine(url_etl)

In [14]:
mensajeria_novedadservicio = pd.read_sql_table('mensajeria_novedadesservicio', mensajeria)
mensajeria_novedadservicio

,id,fecha_novedad,tipo_novedad_id,descripcion,servicio_id,es_prueba,mensajero_id
0,4,2023-11-30 05:00:00+00:00,1,A,51,True,7
1,5,2023-11-30 05:00:00+00:00,1,Halo,51,True,7
2,6,2023-11-30 05:00:00+00:00,1,A,51,True,7
3,7,2023-11-30 05:00:00+00:00,1,B,51,True,7
4,8,2023-11-30 05:00:00+00:00,1,A,51,True,7
...,...,...,...,...,...,...,...
5203,5246,2024-08-31 17:56:46.191229+00:00,1,"Facturaron el refrigerante equivocado, se hará...",28455,True,27
5204,5247,2024-08-31 18:21:14.634785+00:00,2,Edte drrvicio lo hace angelo,28464,True,25
5205,5248,2024-08-31 19:55:19.569591+00:00,2,Edte lo hace csrlos,28467,True,25
5206,5249,2024-08-31 19:55:51.942719+00:00,2,Este lohace csrlos,28466,True,25


In [15]:
dim_fechahora = pd.read_sql_table('dim_fechahora', etl_conn)
dim_novedad = pd.read_sql_table('dim_novedad', etl_conn)
dim_mensajero = pd.read_sql_table('dim_mensajero', etl_conn)

dim_fechahora.head()

,key_dim_fechahora,fecha_hora,fecha_hora_key,año,mes,dia,hora,minuto,dia_de_la_semana
0,0,2023-09-19 16:22:18,1,2023.0,9.0,19.0,16.0,22.0,Tuesday
1,1,2023-09-19 16:30:05,2,2023.0,9.0,19.0,16.0,30.0,Tuesday
2,2,2023-09-19 16:35:52,3,2023.0,9.0,19.0,16.0,35.0,Tuesday
3,3,2023-09-19 16:37:54,4,2023.0,9.0,19.0,16.0,37.0,Tuesday
4,4,2023-09-19 16:49:17,5,2023.0,9.0,19.0,16.0,49.0,Tuesday


In [16]:
from etl.transform.transform_hecho_novedad import transform_hecho_novedad

data = {"mensajeria_novedadesservicio": mensajeria_novedadservicio}

fact_novedad = transform_hecho_novedad(
    data,
    dim_fechahora,
    dim_novedad,
    dim_mensajero,
)
fact_novedad

,novedad_id,servicio_id,fk_fecha,fk_tipo_novedad,fk_mensajero,descripcion,cantidad
0,4,51,127012,1,7,A,1
1,5,51,127012,1,7,Halo,1
2,6,51,127012,1,7,A,1
3,7,51,127012,1,7,B,1
4,8,51,127012,1,7,A,1
...,...,...,...,...,...,...,...
5203,5246,28455,132201,1,25,"Facturaron el refrigerante equivocado, se hará...",1
5204,5247,28464,132202,2,23,Edte drrvicio lo hace angelo,1
5205,5248,28467,132203,2,23,Edte lo hace csrlos,1
5206,5249,28466,132204,2,23,Este lohace csrlos,1


In [17]:
print("Total registros:", len(fact_novedad))
print("Sin fecha:", (fact_novedad["fk_fecha"] == -1).sum())
print("Sin tipo novedad:", (fact_novedad["fk_tipo_novedad"] == -1).sum())
print("Sin mensajero:", (fact_novedad["fk_mensajero"] == -1).sum())

Total registros: 5208
Sin fecha: 0
Sin tipo novedad: 0
Sin mensajero: 0


In [18]:
fact_novedad.to_sql("fact_novedad", etl_conn, if_exists="replace", index=False)

208